In [ ]:
# fix relative imports
import os
cwd = os.path.normpath(os.getcwd())
cwd = cwd.split(os.sep)
find = cwd.index("fidelity-phase-tran")
newdir = f"{os.sep}".join(cwd[:find+1])
os.chdir(newdir)

# import known packages
import numpy as np
import pickle
import gzip
import uuid

from matplotlib import pyplot as plt

# import adhoc packages
from qphaset.phases import gstates_to_rdms_matrix, phases_vfield
from qphaset.plotting import plot_grad_g_angle_stream

## General data exporting
Notebook for exporting the phase transition data for external experiments, eg. machine learning.

In [ ]:
"""
Type of filename to use:
filename = 'results/data/{model}_{identifier}.pkl'
"""
# filename = 'results/annni-20spins-64x64-dmrg-20240212.pkl'             # First decent ANNNI 20
# filename = 'results/ising-20spins-64x64-dmrg-20240214.pkl'

# filename = "annni_ext-93346fe8-cf92-4c55-9b47-0cb2b6f25c0c.pkl"        # ANNNI 50, c1=-1, upside down (*)
# filename = "annni_ext-7e477513-fe51-4ee3-9ae3-75766993ae7a.pkl"        # ANNNI 50, c1=h-1, upside down
# filename = "annni_ext-5234439f-02a3-4e3a-8c6a-a5d0a670dd0c.pkl"        # ANNNI 50, c1=-1, upside down, floating detail
# filename = "annni_ext-d02bd810-dece-40b1-8e36-ac6c8164bb46.pkl"        # ANNNI 50, c1=-0.1, upside down, floating detail
# filename = "annni_ext-aba4d4c0-cb6f-454b-b2e4-8babb3be68b6.pkl"        # ANNNI 50, c1=0, upside down, floating detail, 16 x 16
# filename = "annni_ext-0b44047c-884a-42b9-8bfe-bc5beb29be96.pkl"        # ANNNI 30, c1=0, upside down, floating detail, 32 x 32

# Effects of the grid coarse on the floating phase.
# filename = "annni_ext-b00ffcec-f325-4f18-b5d8-e15111ee72fb.pkl"            # ANNNI 30, c1=-0.1, upside down, floating detail, 32 x 32 (*)
# filename = "annni_ext-a76f14e2-354c-4588-a698-5d7cdaef11ec.pkl"            # ANNNI 30, c1=-0.1, upside down, floating detail, 64 x 64 (*)
filename = "annni_ext-c47e7c80-227f-4b18-b8a4-c910cc6d6908.pkl"            # ANNNI 50, c1=-1, upside down, floating detail, 64 x 64 (**)
# filename = "annni_ext-b9e5a93e-f105-411d-8013-b3120de2b88e.pkl"            # ANNNI 50, c1=-1, upside down, floating detail + multi-critical point, 64 x 64 (**)

# filename = "annni_ext-516fe606-16d0-4bce-afd0-5af6fb262d91.pkl"              # ANNNI 50, c1=-1, upside down, floating detail ++ (**)
# filename = "annni_ext-fad679a7-c06a-4112-8f28-97c92414d43b.pkl"                # ANNNI 30, c1=-1, upside down, floating detail ++ (**)

# filename = "annni_ext-297aa19b-e072-4ec5-b9fc-718bf75db45e.pkl"             # ANNNI 16, c1=-1, upside down, 64 x 64
# filename = "annni_ext-e810f2eb-6837-48a9-a2c7-7acd305f5e70.pkl"             # ANNNI 20, c1=-1, upside down, 64 x 64 (*)


with gzip.open(filename, 'rb') as f:
    data = pickle.load(f)
params = data['params']
l, n = data['l'], data['n']
gstates = data['gstates']
stats = data['stats']

params_extent = np.concatenate([np.min(params, axis=0), np.max(params, axis=0)])
params_extent = tuple(params_extent[[0, 2, 1, 3]])

# sites = [(l // 2) - 2, (l // 2) - 1, l // 2, (l // 2) +1, (l // 2) + 2]
sites = [(l // 2) - 1, l // 2, (l // 2) +1]
# sites = [l // 2]

rdms = gstates_to_rdms_matrix(gstates, sites=sites)

In [ ]:
rdms = rdms[::-1]
g = phases_vfield(rdms, scale=1, grad=False)
grad_g = phases_vfield(rdms, scale=1)

In [ ]:
# Export data for learning

g_params = params.reshape((n, n, -1))[1:-1,1:-1]
assert g_params.shape[:2] == grad_g.shape[:2]

# Export the function we cvall g (and its gradient) in the paper.
filename = f'phaset-data-{str(uuid.uuid4())}.pkl' 
with gzip.open(filename, 'wb') as f:
    pickle.dump(dict(params=g_params, grad_g=grad_g, g=g), f)

In [ ]:
plot_grad_g_angle_stream(grad_g, params_extent=params_extent);